### **1. Load and inspect the file**


In [6]:
import pandas as pd
import numpy as np
import nbformat 

labs = pd.read_csv("/Users/are/Development/DA229/PythonHackathon/Python_Hackathon_Sep_2026/cardiac_failure/labs.csv")

print(labs.shape)       # 2,008 rows, 107 columns
display(labs.head())
labs.info()


(2008, 107)


,inpatient_number,body_temperature,pulse,respiration,systolic_blood_pressure,diastolic_blood_pressure,map_value,fio2,creatinine_enzymatic_method,urea,...,measured_residual_base,measured_bicarbonate,carboxyhemoglobin,body_temperature_blood_gas,oxygen_saturation,partial_oxygen_pressure,oxyhemoglobin,anion_gap,free_calcium,total_hemoglobin
0,857781,36.7,87,19,102,64,76.666667,33,108.3,12.55,...,-2.1,21.2,0.4,37.0,97.0,93.0,95.9,17.8,1.14,125.0
1,743087,36.8,95,18,150,70,96.666667,33,62.0,4.29,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,866418,36.5,98,18,102,67,78.666667,33,185.1,15.99,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,775928,36.0,73,19,110,74,86.000000,33,104.8,8.16,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,810128,35.0,88,19,134,62,86.000000,33,83.9,6.86,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Columns: 107 entries, inpatient_number to total_hemoglobin
dtypes: float64(101), int64(6)
memory usage: 1.6 MB


### **2. Check missing values and duplicates**
**Reasoning :**  This file has no duplicate rows or patient IDs. Many lab columns have missing values because a test was not recorded for every patient. Do not replace all those blanks with zero.



In [7]:
missing = labs.isna().sum().sort_values(ascending=False)
print(missing)

print("Duplicate rows:", labs.duplicated().sum())
print("Duplicate patient IDs:", labs["inpatient_number"].duplicated().sum())


cholinesterase              2008
homocysteine                1862
apolipoprotein_a            1832
apolipoprotein_b            1832
lipoprotein                 1832
                            ... 
diastolic_blood_pressure       0
systolic_blood_pressure        0
respiration                    0
pulse                          0
inpatient_number               0
Length: 107, dtype: int64
Duplicate rows: 0
Duplicate patient IDs: 0


### **3. Remove the completely empty column**
**Reasoning:** The Cholinesterase column is completely empty and hence cannot be used for further analysis. So dropping the column from labs.



In [8]:
empty_columns = labs.columns[labs.isna().all()]
print(empty_columns.tolist())  # ['cholinesterase']

labs = labs.drop(columns=empty_columns)


['cholinesterase']


### **4. Mark unusable vital signs as missing**
**Reasoning:** Three rows have blood pressure recorded as zero; two have systolic pressure below diastolic pressure. Mark the blood pressure values and their calculated map_value as missing in those five rows.



In [9]:
invalid_bp = (
    (labs["systolic_blood_pressure"] == 0) |
    (labs["diastolic_blood_pressure"] == 0) |
    (labs["systolic_blood_pressure"] < labs["diastolic_blood_pressure"])
)

print("Rows with invalid blood pressure:", invalid_bp.sum())

labs.loc[
    invalid_bp,
    ["systolic_blood_pressure", "diastolic_blood_pressure", "map_value"]
] = np.nan

labs.loc[labs["pulse"] == 0, "pulse"] = np.nan
labs.loc[labs["respiration"] == 0, "respiration"] = np.nan


Rows with invalid blood pressure: 5


In [ ]:
keep_cols = [
    "inpatient_number",
    "body_temperature",
    "pulse",
    "respiration",
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
    "fio2",
    "creatinine_enzymatic_method",
    "urea",
    "uric_acid",
    "glomerular_filtration_rate",
    "cystatin",
    "white_blood_cell",
    "monocyte_count",
    "red_blood_cell",
    "hematocrit",
    "lymphocyte_count",
    "hemoglobin",
    "platelet",
    "basophil_count",
    "eosinophil_count",
    "neutrophil_count",
    "activated_partial_thromboplastin_time",
    "fibrinogen",
    "high_sensitivity_troponin",
    "myoglobin",
    "calcium",
    "potassium",
    "chloride",
    "sodium",
    "serum_magnesium",
    "creatine_kinase",
    "lactate_dehydrogenase",
    "brain_natriuretic_peptide",
    "albumin",
    "globulin",
    "total_protein",
    "cholesterol",
    "triglyceride",
    "glucose_blood_gas",
    "lactate",
    "partial_oxygen_pressure",
    "ph",
    "potassium_ion",
    "chloride_ion",
    "sodium_ion",
    "free_calcium"
]



### **Drop these derived / redundant columns**
**<strong style="color: red;">Reasoning:</strong>** 

***1. Derived values should not dominate the dataset***

Columns like:

map_value,
mean_corpuscular_volume,
mean_hemoglobin_volume,
mean_hemoglobin_concentration,
standard_bicarbonate,
anion_gap. 
are typically calculated from more basic variables. If you already have the raw values, these can be recreated later in analysis or feature engineering.

***2. Ratios and percentages are often redundant***

Variables such as:

monocyte_ratio,
basophil_ratio,
eosinophil_ratio,
neutrophil_ratio,
white_globulin_ratio  
are often informative, but they are not always necessary when the absolute count variables are already present. A model can always derive ratios later if needed.

***3. Blood-gas “derived” acid-base variables are not primary features***

These variables are often used for clinical interpretation, but for a simplified modeling dataset they are less useful than the direct measures:

pH,
partial_oxygen_pressure,
partial_pressure_of_carbon_dioxide,
lactate,
glucose_blood_gas 

So keeping the direct values and dropping the computed acid-base summaries is a sensible choice.

***4. Some columns are duplicated in meaning***

Examples:

total_protein and globulin are both part of protein balance,
mean_platelet_volume and platelet are different conceptually, but the calculated index is often less valuable for a reduced dataset,
international_normalized_ratio and prothrombin_time_ratio are related to the same coagulation process.
These are good candidates for dropping when working with a smaller variable set.

***5. A smaller dataset is easier to model and interpret***

Your goal is not to keep every possible field, but to keep:

stable raw measurements,
clinically important markers,
variables that are interpretable without complex calculations,
This makes the dataset easier to clean, visualize, and model without creating noise from highly derived features


In [13]:
drop_cols = [
    "map_value",
    "monocyte_ratio",
    "basophil_ratio",
    "eosinophil_ratio",
    "neutrophil_ratio",
    "mean_corpuscular_volume",
    "mean_hemoglobin_volume",
    "mean_hemoglobin_concentration",
    "mean_platelet_volume",
    "platelet_distribution_width",
    "platelet_hematocrit",
    "coefficient_of_variation_of_red_blood_cell_distribution_width",
    "standard_deviation_of_red_blood_cell_distribution_width",
    "international_normalized_ratio",
    "prothrombin_time_ratio",
    "prothrombin_activity",
    "white_globulin_ratio",
    "standard_residual_base",
    "measured_residual_base",
    "standard_bicarbonate",
    "measured_bicarbonate",
    "total_carbon_dioxide",
    "anion_gap",
    "oxygen_saturation",
    "oxyhemoglobin",
    "carboxyhemoglobin",
    "methemoglobin",
    "hematocrit_blood_gas",
    "total_hemoglobin",
    "hydroxybutyrate_dehydrogenase_to_lactate_dehydrogenase",
    "creatine_kinase_isoenzyme_to_creatine_kinase",
    "total_bile_acid",
    "total_protein",
    "globulin"
]

In [14]:
labs = labs.drop(columns=drop_cols, errors="ignore")

### **Rename the columns for the reduced labs dataframe**
**<strong style="color: red;">Reasoning:</strong>** Keep the names short, consistent, and clinically readable. This helps later modeling and makes the notebook easier to follow. For example: 

**Short and readable:** systolic_bp, diastolic_bp, gfr, wbc, po2\
**Consistent format:** all names are lowercase snake_case\
**Medical convention:** aptt, bnp, ck, ldh, hs_troponin\
**Clearer for analysis:** patient_id instead of inpatient_number




In [15]:
rename_map = {
    "inpatient_number": "patient_id",
    "body_temperature": "body_temp",
    "pulse": "pulse",
    "respiration": "respiration",
    "systolic_blood_pressure": "systolic_bp",
    "diastolic_blood_pressure": "diastolic_bp",
    "fio2": "fio2",
    "creatinine_enzymatic_method": "creatinine",
    "urea": "urea",
    "uric_acid": "uric_acid",
    "glomerular_filtration_rate": "gfr",
    "cystatin": "cystatin",
    "white_blood_cell": "wbc",
    "monocyte_count": "monocyte_count",
    "red_blood_cell": "rbc",
    "hematocrit": "hematocrit",
    "lymphocyte_count": "lymphocyte_count",
    "hemoglobin": "hemoglobin",
    "platelet": "platelet_count",
    "basophil_count": "basophil_count",
    "eosinophil_count": "eosinophil_count",
    "neutrophil_count": "neutrophil_count",
    "activated_partial_thromboplastin_time": "aptt",
    "fibrinogen": "fibrinogen",
    "high_sensitivity_troponin": "hs_troponin",
    "myoglobin": "myoglobin",
    "calcium": "calcium",
    "potassium": "potassium",
    "chloride": "chloride",
    "sodium": "sodium",
    "serum_magnesium": "magnesium",
    "creatine_kinase": "ck",
    "lactate_dehydrogenase": "ldh",
    "brain_natriuretic_peptide": "bnp",
    "albumin": "albumin",
    "globulin": "globulin",
    "total_protein": "total_protein",
    "cholesterol": "cholesterol",
    "triglyceride": "triglyceride",
    "glucose_blood_gas": "glucose_blood_gas",
    "lactate": "lactate",
    "partial_oxygen_pressure": "po2",
    "ph": "ph",
    "potassium_ion": "potassium_ion",
    "chloride_ion": "chloride_ion",
    "sodium_ion": "sodium_ion",
    "free_calcium": "free_calcium"
}

labs = labs.rename(columns=rename_map)

In [21]:
labs.describe()
labs.isna().sum()
display(labs.dtypes)

patient_id                      int64
body_temp                     float64
pulse                         float64
respiration                   float64
systolic_bp                   float64
                               ...   
glucose_blood_gas             float64
lactate                       float64
body_temperature_blood_gas    float64
po2                           float64
free_calcium                  float64
Length: 72, dtype: object